# Diffusion Variants and Practical Engineering

Once the core DDPM pipeline is in place, the next questions are practical rather than foundational: how can sampling be made faster, how can generation become more controllable, how can diffusion be moved into a latent space, and how can evaluation avoid unnecessary repeated work?


The focus here is not only “what are the equations?” but “what changes when one wants a system that is **faster**, **more controllable**, or **more scalable**?” Some of the code below is intentionally schematic, but the engineering logic is real.


```{important}
This discussion is about **system design**. Once the base denoiser works, performance depends heavily on the sampler, the conditioning mechanism, the latent space, and the evaluation pipeline.
```


## Shared Setup for Practical Variants

A small setup cell keeps the practical samplers and metric helpers readable as real code rather than as floating pseudocode. These snippets assume access to a trained DDPM-style denoiser.


In [ ]:
import math
import copy
import torch
import torch.nn.functional as F
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
timesteps = 300


def make_ddpm_schedule(timesteps, device):
    betas = torch.linspace(1e-4, 0.02, timesteps, device=device)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    alpha_bars_prev = torch.cat([torch.tensor([1.0], device=device), alpha_bars[:-1]], dim=0)
    posterior_variance = betas * (1.0 - alpha_bars_prev) / (1.0 - alpha_bars)
    return {
        "betas": betas,
        "alphas": alphas,
        "alpha_bars": alpha_bars,
        "posterior_variance": posterior_variance,
    }


schedule = make_ddpm_schedule(timesteps, device)


## DDIM as a Practical Sampler

The first practical upgrade is **DDIM**. The denoiser is unchanged, but the sampling trajectory becomes partially or fully deterministic. This matters because the main cost of diffusion is not usually training stability; it is the number of reverse evaluations required at inference time.


In [ ]:
def extract(coefficients, t, x_shape):
    out = coefficients.gather(0, t)
    return out.view(t.shape[0], *((1,) * (len(x_shape) - 1)))


@torch.no_grad()
def ddim_sample(model, shape, ddim_steps=50, eta=0.0, device="cpu"):
    betas = schedule["betas"]
    alpha_bars = schedule["alpha_bars"]
    step_indices = torch.linspace(0, betas.size(0) - 1, ddim_steps, device=device).long()
    x = torch.randn(shape, device=device)

    for idx in reversed(range(ddim_steps)):
        t_scalar = step_indices[idx].item()
        t = torch.full((shape[0],), t_scalar, device=device, dtype=torch.long)
        predicted_noise = model(x, t)

        alpha_bar_t = extract(alpha_bars, t, x.shape)
        x0_hat = (x - torch.sqrt(1.0 - alpha_bar_t) * predicted_noise) / torch.sqrt(alpha_bar_t)

        if idx == 0:
            x = x0_hat
            continue

        prev_t = torch.full((shape[0],), step_indices[idx - 1].item(), device=device, dtype=torch.long)
        alpha_bar_prev = extract(alpha_bars, prev_t, x.shape)
        sigma_t = (
            eta
            * torch.sqrt((1 - alpha_bar_prev) / (1 - alpha_bar_t))
            * torch.sqrt(1 - alpha_bar_t / alpha_bar_prev)
        )
        direction = torch.sqrt(torch.clamp(1 - alpha_bar_prev - sigma_t**2, min=0.0)) * predicted_noise
        noise = sigma_t * torch.randn_like(x) if eta > 0 else 0.0
        x = torch.sqrt(alpha_bar_prev) * x0_hat + direction + noise

    return x


The practical point is simple: DDIM is one of the first places where diffusion stops being “just the model” and becomes **model plus sampler design**. The denoiser may stay fixed while the sampling path changes substantially.


## Classifier-Free Guidance as a Control Mechanism

Once a denoiser can run with and without conditioning, one can mix the two predictions during sampling. This is the practical heart of **classifier-free guidance**. The unconditional branch says “move toward a plausible image.” The conditional branch says “move toward an image matching the prompt or label.” The **guidance scale** decides how strongly to privilege the second over the first.


```{note}
Classifier-free guidance is one of the clearest examples of **conditional generation** becoming a practical control interface. The model is still generative, but generation is no longer purely unconditional.
```


In [ ]:
@torch.no_grad()
def guided_noise_prediction(model, xt, t, cond, guidance_scale):
    eps_uncond = model(xt, t, None)
    eps_cond = model(xt, t, cond)
    return eps_uncond + guidance_scale * (eps_cond - eps_uncond)


@torch.no_grad()
def guided_ddpm_step(model, xt, t, cond, guidance_scale, schedule):
    betas = schedule["betas"]
    alphas = schedule["alphas"]
    alpha_bars = schedule["alpha_bars"]
    posterior_variance = schedule["posterior_variance"]

    beta_t = extract(betas, t, xt.shape)
    alpha_t = extract(alphas, t, xt.shape)
    alpha_bar_t = extract(alpha_bars, t, xt.shape)
    predicted_noise = guided_noise_prediction(model, xt, t, cond, guidance_scale)

    mean = (xt - beta_t * predicted_noise / torch.sqrt(1.0 - alpha_bar_t)) / torch.sqrt(alpha_t)
    variance = extract(posterior_variance, t, xt.shape)
    noise = torch.randn_like(xt)
    return mean + torch.sqrt(variance) * noise


## Latent Diffusion as a Systems Choice

Large image systems rarely run diffusion directly in raw pixel space. They often run it in the **latent space** of an autoencoder. This is not a cosmetic change. It is a **systems-level decision** that changes the memory footprint, the sampler cost, and often the semantic smoothness of the representation.


In [ ]:
class LatentDiffusionPipeline:
    def __init__(self, autoencoder, denoiser, schedule):
        self.autoencoder = autoencoder
        self.denoiser = denoiser
        self.schedule = schedule

    @torch.no_grad()
    def encode_images(self, images):
        return self.autoencoder.encode(images)

    @torch.no_grad()
    def decode_latents(self, latents):
        return self.autoencoder.decode(latents)

    @torch.no_grad()
    def sample_latents(self, shape, cond=None, guidance_scale=1.0):
        latents = torch.randn(shape, device=device)
        for t_scalar in reversed(range(timesteps)):
            t = torch.full((shape[0],), t_scalar, device=device, dtype=torch.long)
            if cond is None:
                eps = self.denoiser(latents, t)
            else:
                eps = guided_noise_prediction(
                    self.denoiser,
                    latents,
                    t,
                    cond,
                    guidance_scale,
                )
            latents = guided_ddpm_step(
                self.denoiser,
                latents,
                t,
                cond,
                guidance_scale,
                self.schedule,
            )
        return latents


## Cached FID/KID Evaluation

Diffusion evaluation is expensive enough that recomputing real-image features repeatedly is pure waste. The same caching pattern becomes especially valuable here because samplers are slow and comparisons among several samplers or guidance settings are often needed.


In [ ]:
def prepare_for_inception_metrics(images):
    if images.size(1) == 1:
        images = images.repeat(1, 3, 1, 1)
    return images.clamp(0.0, 1.0)


@torch.no_grad()
def compute_cached_diffusion_metrics(sample_fn, real_loader, device, num_samples=1000, metric_batch_size=32):
    fid = FrechetInceptionDistance(
        feature=2048,
        normalize=True,
        reset_real_features=False,
    ).set_dtype(torch.float64).to(device)
    kid = KernelInceptionDistance(
        feature=2048,
        subsets=10,
        subset_size=100,
        normalize=True,
        reset_real_features=False,
    ).to(device)

    seen_real = 0
    for real_images, _ in tqdm(real_loader, desc="Diffusion real metrics", leave=False):
        remaining = num_samples - seen_real
        if remaining <= 0:
            break
        real_images = real_images[: min(metric_batch_size, remaining)].to(device)
        real_images = prepare_for_inception_metrics(real_images)
        fid.update(real_images, real=True)
        kid.update(real_images, real=True)
        seen_real += real_images.size(0)

    generated = 0
    pbar = tqdm(total=num_samples, desc="Diffusion fake metrics", leave=False)
    while generated < num_samples:
        batch_n = min(metric_batch_size, num_samples - generated)
        fake_images = sample_fn(batch_n).to(device)
        fake_images = prepare_for_inception_metrics(fake_images)
        fid.update(fake_images, real=False)
        kid.update(fake_images, real=False)
        generated += batch_n
        pbar.update(batch_n)
    pbar.close()

    kid_mean, kid_std = kid.compute()
    return {
        "fid": fid.compute().item(),
        "kid_mean": kid_mean.item(),
        "kid_std": kid_std.item(),
    }


## Practical Perspective

The core ideas at this stage are straightforward: the denoiser may stay fixed while the **sampler** changes, conditioning may be strengthened through **guidance**, image generation may be moved into a **latent space**, and repeated quantitative comparison benefits from **cached evaluation**. These are the main practical levers that turn a basic diffusion model into a more usable system.
